# 🏀 EDA — La France, locomotive du basket européen en NBA
**Mission DATA | Wild Code School**  
Sources : `all_seasons.csv` (1996–2023) + `nba_api` (2023–2025)

---
> *Ce notebook charge d'abord toutes les données (CSV + API), les fusionne,  
> puis réalise l'exploration complète sur le dataset final.*


## 1. Imports — Les bibliothèques nécessaires
On commence par importer toutes les bibliothèques qu'on va utiliser dans ce notebook.


In [16]:
import pandas as pd
import numpy as np

print('Bibliotheques importees !')

Bibliotheques importees !


## 2. Chargement du CSV historique (1996–2023)
On charge `all_seasons.csv` qui contient toutes les saisons de 1996 à 2022-23.  
Un **DataFrame** c'est comme un tableau Excel en Python.


In [17]:
df_hist = pd.read_csv('all_seasons.csv', index_col=0)  # index_col=0 supprime Unnamed: 0

print(f"{df_hist.shape[0]} lignes x {df_hist.shape[1]} colonnes")
print(f"Saisons : {df_hist['season'].min()} -> {df_hist['season'].max()}")
print(f"Joueurs francais : {(df_hist['country'] == 'France').sum()}")

12844 lignes x 21 colonnes
Saisons : 1996-97 -> 2022-23
Joueurs francais : 190


## 3. Chargement des saisons recentes (2023-2025)
On charge les deux CSV supplementaires pour les saisons manquantes.  
Place `nba_2023_24.csv` et `nba_2024_25.csv` dans le meme dossier que ce notebook.

In [18]:
# Chargement des deux CSV recents (sans index_col=0 : leur 1ere colonne est player_name, pas un index)
df_2324 = pd.read_csv('nba_2023_24.csv')  # saison 2023-24
df_2425 = pd.read_csv('nba_2024_25.csv')  # saison 2024-25

# Ajout de la colonne saison si elle n'est pas dans le CSV
if 'season' not in df_2324.columns:
    df_2324['season'] = '2023-24'
if 'season' not in df_2425.columns:
    df_2425['season'] = '2024-25'

# Fusion des deux saisons en un seul DataFrame
df_recent = pd.concat([df_2324, df_2425], ignore_index=True)

print(f'Saison 2023-24 : {len(df_2324)} joueurs')
print(f'Saison 2024-25 : {len(df_2425)} joueurs')
print(f'Total : {df_recent.shape[0]} lignes x {df_recent.shape[1]} colonnes')
print(f'Joueurs francais 2023-25 : {(df_recent["country"] == "France").sum()}')

Saison 2023-24 : 37 joueurs
Saison 2024-25 : 33 joueurs
Total : 70 lignes x 21 colonnes
Joueurs francais 2023-25 : 17


## 4. Fusion CSV historique + CSV recents → Dataset final
On combine les donnees historiques et recentes en **un seul DataFrame**.  
`pd.concat` empile les lignes des deux sources.

In [19]:
# On garde seulement les colonnes communes aux deux sources
cols = [c for c in df_hist.columns if c in df_recent.columns]

# Fusion CSV historique + CSV recents en un seul dataset
df = pd.concat([df_hist, df_recent[cols]], ignore_index=True)

print(f"Dataset : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"Periode : {df['season'].min()} -> {df['season'].max()}")
print(f"Joueurs francais : {(df['country'] == 'France').sum()}")

Dataset : 12914 lignes x 21 colonnes
Periode : 1996-97 -> 2024-25
Joueurs francais : 207


## 5. Dimensions du dataset final
On vérifie la taille exacte de notre dataset complet.


In [20]:
print(f"{df.shape[0]} lignes x {df.shape[1]} colonnes")
df.head()

12914 lignes x 21 colonnes


,player_name,team_abbreviation,age,player_height,player_weight,college,country,draft_year,draft_round,draft_number,...,pts,reb,ast,net_rating,oreb_pct,dreb_pct,usg_pct,ts_pct,ast_pct,season
0,Randy Livingston,HOU,22.0,193.04,94.800728,Louisiana State,USA,1996,2,42,...,3.9,1.5,2.4,0.3,0.042,0.071,0.169,0.487,0.248,1996-97
1,Gaylon Nickerson,WAS,28.0,190.50,86.182480,Northwestern Oklahoma,USA,1994,2,34,...,3.8,1.3,0.3,8.9,0.030,0.111,0.174,0.497,0.043,1996-97
2,George Lynch,VAN,26.0,203.20,103.418976,North Carolina,USA,1993,1,12,...,8.3,6.4,1.9,-8.2,0.106,0.185,0.175,0.512,0.125,1996-97
3,George McCloud,LAL,30.0,203.20,102.058200,Florida State,USA,1989,1,7,...,10.2,2.8,1.7,-2.7,0.027,0.111,0.206,0.527,0.125,1996-97
4,George Zidek,DEN,23.0,213.36,119.748288,UCLA,USA,1995,1,22,...,2.8,1.7,0.3,-14.1,0.102,0.169,0.195,0.500,0.064,1996-97


## 6. Types de données par colonne
Chaque colonne a un type : `object` = texte, `float64` = décimal, `int64` = entier.


In [21]:
print("=== Types des colonnes ===")
print(df.dtypes)   # affiche le type de chaque colonne


=== Types des colonnes ===
player_name              str
team_abbreviation        str
age                  float64
player_height        float64
player_weight        float64
college               object
country                  str
draft_year            object
draft_round           object
draft_number          object
gp                     int64
pts                  float64
reb                  float64
ast                  float64
net_rating           float64
oreb_pct             float64
dreb_pct             float64
usg_pct              float64
ts_pct               float64
ast_pct              float64
season                   str
dtype: object


## 7. Valeurs manquantes (NaN)
**NaN** = case vide. On identifie où sont les trous dans notre dataset.


In [22]:
nulls = df.isnull().sum()
print(nulls[nulls > 0])                  # seulement les colonnes avec NaN
print(f"Total NaN : {nulls.sum()}")

college         1924
draft_round        1
draft_number       1
dtype: int64
Total NaN : 1926


## 8. Doublons
Un doublon = ligne identique à une autre. On les compte (sans supprimer pour l'instant).


In [23]:
print(f"Doublons : {df.duplicated().sum()}")

Doublons : 0


## 9. Statistiques descriptives
`describe()` calcule automatiquement moyenne, min, max, quartiles pour chaque colonne numérique.


In [24]:
df.describe().round(2)   # stats descriptives arrondies à 2 décimales


,age,player_height,player_weight,gp,pts,reb,ast,net_rating,oreb_pct,dreb_pct,usg_pct,ts_pct,ast_pct
count,12914.00,12914.00,12914.00,12914.00,12914.00,12914.00,12914.00,12914.00,12914.00,12914.00,12914.00,12914.00,12914.00
mean,27.04,200.58,100.27,51.20,8.25,3.57,1.83,-2.21,0.05,0.14,0.18,0.51,0.13
std,4.34,9.12,12.43,25.05,6.05,2.49,1.81,12.64,0.04,0.06,0.05,0.10,0.09
min,18.00,160.02,60.33,1.00,0.00,0.00,0.00,-250.00,0.00,0.00,0.00,0.00,0.00
25%,24.00,193.04,90.72,31.00,3.60,1.80,0.60,-6.30,0.02,0.10,0.15,0.48,0.07
50%,26.00,200.66,99.79,57.00,6.70,3.00,1.20,-1.30,0.04,0.13,0.18,0.53,0.10
75%,30.00,208.28,108.86,73.00,11.50,4.70,2.40,3.20,0.08,0.18,0.22,0.56,0.18
max,44.00,231.14,163.29,85.00,36.10,16.30,11.70,300.00,1.00,1.00,1.00,1.50,1.00


## 10. Distribution par pays — La France dans le monde NBA 🇫🇷
On regarde la répartition des joueurs par pays d'origine sur toute la période.


In [25]:
pays_counts = df['country'].value_counts()
print(pays_counts.head(15))

rang_france = pays_counts.index.get_loc('France') + 1
print(f"France : rang #{rang_france} | {pays_counts['France']} entrees")

country
USA          10721
Canada         214
France         207
Australia      105
Spain           95
Brazil          87
Turkey          79
Slovenia        78
Serbia          77
Croatia         73
Germany         73
Argentina       67
Lithuania       66
Ukraine         45
Nigeria         43
Name: count, dtype: int64
France : rang #3 | 207 entrees


In [26]:
# Top 10 pays hors USA
pays_hors_usa = pays_counts.drop('USA')    # enleve les USA
top10 = pays_hors_usa.head(10)             # top 10

print("=== Top 10 nations etrangeres en NBA (hors USA) ===")
print(top10)

=== Top 10 nations etrangeres en NBA (hors USA) ===
country
Canada       214
France       207
Australia    105
Spain         95
Brazil        87
Turkey        79
Slovenia      78
Serbia        77
Croatia       73
Germany       73
Name: count, dtype: int64


## 11. Focus joueurs français 🇫🇷
On filtre le dataset sur `country == 'France'` et on analyse leurs performances.


In [27]:
df_france = df[df['country'] == 'France']

print(f"{len(df_france)} entrees | {df_france['player_name'].nunique()} joueurs uniques")
for j in sorted(df_france['player_name'].dropna().unique()):  # dropna() enleve les NaN avant le tri
    print(f'  - {j}')

207 entrees | 46 joueurs uniques
  - Adam Mokoka
  - Alexandre Sarr
  - Alexis Ajinca
  - Axel Toupane
  - Bilal Coulibaly
  - Boris Diaw
  - Damien Inglis
  - Elie Okobo
  - Evan Fournier
  - Frank Ntilikina
  - Guerschon Yabusele
  - Ian Mahinmi
  - Jaylen Hoard
  - Jerome Moiso
  - Joel Ayayi
  - Joffrey Lauvergne
  - Johan Petro
  - Kevin Seraphin
  - Killian Hayes
  - Killian Tillie
  - Mickael Pietrus
  - Moussa Diabate
  - Nando De Colo
  - Nicolas Batum
  - Olivier Sarr
  - Ousmane Dieng
  - Pacome Dadiet
  - Pape Sy
  - Petr Cornelie
  - Rayan Rupert
  - Rodrigue Beaubois
  - Ronny Turiaf
  - Rudy Gobert
  - Sekou Doumbouya
  - Sidy Cissoko
  - Tariq Abdul-Wahad
  - Theo Maledon
  - Tidjane Salaun
  - Timothe Luwawu-Cabarrot
  - Tony Parker
  - Victor Wembanyama
  - Vincent Poirier
  - William Howard
  - Yves Missi
  - Yves Pons
  - Zaccharie Risacher


In [28]:
top_pts = df_france.groupby('player_name')['pts'].mean().sort_values(ascending=False).head(10).round(2)
print(top_pts)

player_name
Victor Wembanyama     22.95
Tony Parker           15.37
Zaccharie Risacher    13.50
Evan Fournier         12.86
Rudy Gobert           12.42
Alexandre Sarr        10.20
Nicolas Batum         10.04
Bilal Coulibaly        9.80
Boris Diaw             8.42
Killian Hayes          8.00
Name: pts, dtype: float64


In [29]:
# Evolution du nombre de joueurs francais par saison
fr_par_saison = df_france.groupby('season')['player_name'].nunique()   # nb joueurs / saison

print("=== Nb de joueurs francais par saison ===")
print(fr_par_saison)

=== Nb de joueurs francais par saison ===
season
1997-98     1
1998-99     1
1999-00     1
2000-01     2
2001-02     3
2002-03     3
2003-04     4
2004-05     4
2005-06     5
2006-07     5
2007-08     6
2008-09     7
2009-10     9
2010-11    11
2011-12     9
2012-13    11
2013-14    10
2014-15    10
2015-16    11
2016-17    11
2017-18     9
2018-19     9
2019-20    12
2020-21    13
2021-22    14
2022-23     9
2023-24     7
2024-25    10
Name: player_name, dtype: int64


## 12. Valeurs aberrantes (Outliers)
Un outlier = valeur anormalement grande ou petite. On compare moyenne vs min/max.


In [30]:
df[['pts', 'reb', 'ast', 'gp', 'age']].describe().round(2)

,pts,reb,ast,gp,age
count,12914.00,12914.00,12914.00,12914.00,12914.00
mean,8.25,3.57,1.83,51.20,27.04
std,6.05,2.49,1.81,25.05,4.34
min,0.00,0.00,0.00,1.00,18.00
25%,3.60,1.80,0.60,31.00,24.00
50%,6.70,3.00,1.20,57.00,26.00
75%,11.50,4.70,2.40,73.00,30.00
max,36.10,16.30,11.70,85.00,44.00


---
# 🧹 ÉTAPE 3 — Nettoyage des données

**Objectif :** UN seul dataset propre à charger dans Power BI.

**Plan :**
1. Supprimer la colonne `college` (inutile pour l'analyse)
2. Corriger les doublons joueur+saison (multi-équipes)
3. Harmoniser `draft_round` / `draft_number`
4. Filtrer les outliers `net_rating`
5. Ajouter les colonnes analytiques (`continent`, `is_french`, `is_european`)
6. Export `nba_dataset_clean.csv`


## 3.1 Supprimer la colonne `college`
Cette colonne est vide pour la majorité des joueurs internationaux.  
Elle n'apporte rien à l'analyse → on la supprime.


In [31]:
print(f"Colonnes avant : {list(df.columns)}")   # liste des colonnes actuelles

df = df.drop(columns=['college'])             # supprime la colonne college

print(f"Colonnes après : {list(df.columns)}")  # vérification
print(f"Shape : {df.shape}")                   # nouvelles dimensions


Colonnes avant : ['player_name', 'team_abbreviation', 'age', 'player_height', 'player_weight', 'college', 'country', 'draft_year', 'draft_round', 'draft_number', 'gp', 'pts', 'reb', 'ast', 'net_rating', 'oreb_pct', 'dreb_pct', 'usg_pct', 'ts_pct', 'ast_pct', 'season']
Colonnes après : ['player_name', 'team_abbreviation', 'age', 'player_height', 'player_weight', 'country', 'draft_year', 'draft_round', 'draft_number', 'gp', 'pts', 'reb', 'ast', 'net_rating', 'oreb_pct', 'dreb_pct', 'usg_pct', 'ts_pct', 'ast_pct', 'season']
Shape : (12914, 20)


## 3.2 Doublons joueur + saison
Certains joueurs ont été tradés en cours de saison → deux lignes pour le même joueur+saison.  
**Règle :** on garde la ligne avec le plus grand nombre de matchs joués (`gp`).


In [32]:
# Identifier les doublons joueur+saison
doublons_mask = df.duplicated(subset=['player_name', 'season'], keep=False)  # marque toutes les lignes dupliquées
doublons = df[doublons_mask]                                                   # extrait les doublons

print(f"Lignes en doublon : {len(doublons)}")
print(doublons[['player_name', 'team_abbreviation', 'season', 'gp']].to_string())  # affiche les cas


Lignes en doublon : 8
          player_name team_abbreviation   season  gp
5163  Marcus Williams               LAC  2007-08  11
5164  Marcus Williams               NJN  2007-08  53
5635  Marcus Williams               GSW  2008-09   9
5643  Marcus Williams               SAS  2008-09   2
7291    Chris Johnson               MIN  2012-13  30
7292    Chris Johnson               MEM  2012-13   8
7962    Tony Mitchell               MIL  2013-14   3
7963    Tony Mitchell               DET  2013-14  21


In [33]:
# On trie par gp décroissant puis on garde la 1ère ligne par joueur+saison
df = df.sort_values('gp', ascending=False)                                      # plus grand gp en premier
df = df.drop_duplicates(subset=['player_name', 'season'], keep='first')         # garde max gp
df = df.reset_index(drop=True)                                                  # réindexe de 0 à n

print(f"Shape après suppression doublons : {df.shape}")
print(f"Doublons restants : {df.duplicated(subset=['player_name','season']).sum()}")  # doit être 0


Shape après suppression doublons : (12910, 20)
Doublons restants : 0


## 3.3 Harmoniser `draft_round` et `draft_number`
Ces colonnes mélangent entiers, floats et strings.  
On standardise : `'1'`, `'2'`, `'Undrafted'`.


In [34]:
print("=== draft_round AVANT ===")
print(df['draft_round'].value_counts().head(10))   # valeurs actuelles


=== draft_round AVANT ===
draft_round
1            7351
2            3030
Undrafted    2409
1.0            53
3              20
2.0            16
4              12
0               6
7               5
6               5
Name: count, dtype: int64


In [35]:
# Étape 1 : tout convertir en string
df['draft_round']  = df['draft_round'].astype(str)    # texte
df['draft_number'] = df['draft_number'].astype(str)   # texte

# Étape 2 : supprimer les '.0' parasites ('1.0' → '1')
df['draft_round']  = df['draft_round'].str.replace('.0', '', regex=False)
df['draft_number'] = df['draft_number'].str.replace('.0', '', regex=False)

# Étape 3 : 'nan' → 'Undrafted'
df['draft_round']  = df['draft_round'].replace('nan', 'Undrafted')
df['draft_number'] = df['draft_number'].replace('nan', 'Undrafted')

# Étape 4 : valeurs > 2 dans draft_round sont des erreurs → 'Undrafted'
valides = ['1', '2', 'Undrafted']
df['draft_round'] = df['draft_round'].apply(lambda x: x if x in valides else 'Undrafted')

print("=== draft_round APRÈS ===")
print(df['draft_round'].value_counts())   # vérification


=== draft_round APRÈS ===
draft_round
1            7404
2            3046
Undrafted    2460
Name: count, dtype: int64


## 3.4 Filtrer les outliers `net_rating`
Valeurs < -50 ou > 50 = joueurs avec très peu de minutes → non représentatifs.  
On les retire pour ne pas biaiser les moyennes.


In [36]:
print(f"Outliers net_rating < -50 : {(df['net_rating'] < -50).sum()}")
print(f"Outliers net_rating > +50 : {(df['net_rating'] >  50).sum()}")
print(f"Shape avant : {df.shape}")

df = df[(df['net_rating'] >= -50) & (df['net_rating'] <= 50)]  # filtre les extrêmes
df = df.reset_index(drop=True)                                   # réindexe

print(f"Shape après  : {df.shape}")
print(f"net_rating → min: {df['net_rating'].min()} | max: {df['net_rating'].max()}")


Outliers net_rating < -50 : 94
Outliers net_rating > +50 : 35
Shape avant : (12910, 20)
Shape après  : (12781, 20)
net_rating → min: -50.0 | max: 50.0


## 3.5 Colonnes analytiques
On crée trois colonnes qui serviront directement dans Power BI et les analyses des 3 niveaux :
- `is_french` → True si le joueur est français  
- `is_european` → True si le joueur vient d'Europe  
- `continent` → continent d'origine


In [37]:
# Liste des pays européens présents dans le dataset
pays_europeens = [
    'France', 'Serbia', 'Slovenia', 'Greece', 'Germany', 'Croatia', 'Spain',
    'Latvia', 'Lithuania', 'Bosnia', 'Montenegro', 'Ukraine', 'Georgia',
    'Turkey', 'Czech Republic', 'Poland', 'Sweden', 'Finland', 'Italy',
    'Portugal', 'Netherlands', 'Belgium', 'Switzerland', 'Austria',
    'Hungary', 'Romania', 'Russia', 'Belarus', 'Slovakia', 'Norway',
    'Denmark', 'Israel'
]

pays_am_nord  = ['USA', 'Canada', 'Mexico']
pays_am_sud   = ['Brazil', 'Argentina', 'Venezuela', 'Dominican Republic',
                  'Puerto Rico', 'Colombia', 'Uruguay']
pays_oceanie  = ['Australia', 'New Zealand']
pays_afrique  = ['Nigeria', 'Cameroon', 'Senegal', 'Mali', 'Congo',
                  'South Sudan', 'Angola', 'Ivory Coast', 'Ghana']
pays_asie     = ['China', 'Japan', 'South Korea', 'Philippines']

# is_french : True si France
df['is_french'] = df['country'] == 'France'             # booléen

# is_european : True si pays européen
df['is_european'] = df['country'].isin(pays_europeens)  # booléen

# continent : catégorie
def get_continent(country):                              # fonction de mapping
    if country in pays_europeens : return 'Europe'
    if country in pays_am_nord   : return 'Amérique du Nord'
    if country in pays_am_sud    : return 'Amérique du Sud'
    if country in pays_oceanie   : return 'Océanie'
    if country in pays_afrique   : return 'Afrique'
    if country in pays_asie      : return 'Asie'
    return 'Autre'

df['continent'] = df['country'].apply(get_continent)    # applique la fonction

print("=== Répartition par continent ===")
print(df['continent'].value_counts())
print()
print(f"Joueurs français    : {df['is_french'].sum()} entrées")
print(f"Joueurs européens   : {df['is_european'].sum()} entrées")


=== Répartition par continent ===
continent
Amérique du Nord    10839
Europe               1134
Autre                 279
Amérique du Sud       226
Afrique               148
Océanie               125
Asie                   30
Name: count, dtype: int64

Joueurs français    : 205 entrées
Joueurs européens   : 1134 entrées


## 3.6 Résumé — Dataset propre ✅


In [38]:
print("=" * 55)
print("  RÉSUMÉ NETTOYAGE — nba_dataset_clean")
print("=" * 55)
print(f"  Shape             : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"  Valeurs manquantes: {df.isnull().sum().sum()}")
print(f"  Doublons          : {df.duplicated().sum()}")
print(f"  Saisons           : {df['season'].min()} → {df['season'].max()}")
print(f"  Pays              : {df['country'].nunique()}")
print(f"  Joueurs français  : {df['is_french'].sum()} entrées | {df[df['is_french']]['player_name'].nunique()} joueurs uniques")
print(f"  Joueurs européens : {df['is_european'].sum()} entrées")
print()
print("  Colonnes finales :")
for c in df.columns:
    print(f"    - {c}")


  RÉSUMÉ NETTOYAGE — nba_dataset_clean
  Shape             : 12781 lignes × 23 colonnes
  Valeurs manquantes: 1
  Doublons          : 0
  Saisons           : 1996-97 → 2024-25
  Pays              : 82
  Joueurs français  : 205 entrées | 45 joueurs uniques
  Joueurs européens : 1134 entrées

  Colonnes finales :
    - player_name
    - team_abbreviation
    - age
    - player_height
    - player_weight
    - country
    - draft_year
    - draft_round
    - draft_number
    - gp
    - pts
    - reb
    - ast
    - net_rating
    - oreb_pct
    - dreb_pct
    - usg_pct
    - ts_pct
    - ast_pct
    - season
    - is_french
    - is_european
    - continent


## 3.7 Export `nba_dataset_clean.csv`
> ✅ Accord requis — passe `EXPORTER = True` pour créer le fichier.  
> Ce CSV sera **la seule source** importée dans Power BI.


In [39]:
EXPORTER = False   # ← passe à True pour exporter

if EXPORTER:
    df.to_csv('nba_dataset_clean.csv', index=False)   # export sans index
    print("✅ nba_dataset_clean.csv exporté !")
    print(f"   {df.shape[0]} lignes × {df.shape[1]} colonnes")
    print("   → Importe ce fichier dans Power BI Desktop")
else:
    print("ℹ️  EXPORTER = False — fichier non créé.")
    print("   Mets EXPORTER = True et relance pour exporter.")


ℹ️  EXPORTER = False — fichier non créé.
   Mets EXPORTER = True et relance pour exporter.
